In [ ]:
import numpy as np
import os
import pyvista as pv
import pyacvd
# import centerline_simp as cl_s
from scipy.interpolate import interp1d
from SUPORT_def_deformation import *

# 1. Read stuff first

In [ ]:
# Paper algorithms use the public config interface. All clinical values and paths
# come from git-ignored config/local.py; copy config/local_example.py for its schema.
from config import load_case

# A private label must be explicit. The synthetic driver sets PHANTOM through this
# same environment variable and executes this notebook without editing it.
CASE = os.environ.get("AORTA_CASE")
if not CASE:
    raise RuntimeError("Set AORTA_CASE to a private label or PHANTOM")
cfg = load_case(CASE)

number          = cfg["dataset_id"]
folder_dir_sim  = cfg["sim_folder"]
mean_hr         = cfg["heart_rate_bpm"]
list_ignore     = list(cfg["rings_ignored"])

time_size = cfg["cycle_duration_s"]          # 60 / HR, seconds
print(f"case {CASE}: time size {time_size} seconds")

out_name = ["inlet.vtp", "out.vtp", "out1.vtp", "out2.vtp", "out3.vtp", "interface.vtp"]

dir_stl  = cfg["segmentation_dir"]
main_dir = cfg["mesh_surfaces_dir"] + os.sep
dir_mesh = os.path.join(cfg["mesh_surfaces_dir"], "interface.vtp")
mesh_ref = pv.read(dir_mesh)

dir_mesh_full  = cfg["mesh_exterior"]
dir_centerline = cfg["centerline"]
centerline     = pv.read(dir_centerline)

dir_cuts = cfg["cuts_posit"]
data = read_cuts_posit(dir_cuts)
plane_info_0   = np.array(data[0])
plane_info_arc = np.array(data[1])


In [ ]:
meshes=[]
print("number: ",number)
# cfg["segmentation_dir"] already ends in the dataset identifier, so it must not
# be appended again; os.path.join rather than a literal "\\" so this resolves off
# Windows too.
direct=os.path.join(dir_stl,"Segmentation_AI")
infolder_total=os.listdir(direct)


organized_files={}  
for file_name in infolder_total:
        if file_name.endswith(".stl") and file_name[0].isdigit():
            first_digit = int(file_name.split('_')[0].replace('%', ''))  # Extract the first digit
            organized_files.setdefault(first_digit, []).append(file_name)

load_name_mesh,load_name_mesh_load=[],[] 
    
for digit in sorted(organized_files.keys(), key=lambda x: f"{x:03}"):
    load_name_mesh.append(organized_files[digit][0])
    load_name_mesh_load.append(os.path.join(direct, organized_files[digit][0]))
print(sorted(organized_files.keys()))
for file in load_name_mesh_load:
        mesh_r=pv.read(file)
        mesh_r=mesh_r.clean().fill_holes(50)
        clus = pyacvd.Clustering(mesh_r)
        clus.cluster(10000)
        mesh_rem = clus.create_mesh()
        mesh_rem.fill_holes(1,inplace=True)
        meshes.append(mesh_rem)
        
orginal_0=meshes[0].copy()

if cfg["wall_cycle_closure"] != "replace_last_with_reference":
    raise ValueError(f"unsupported wall cycle closure: {cfg['wall_cycle_closure']}")
# The final supplied phase is the periodic endpoint, not an independently
# retained measurement. Replace it with the reference and map the resulting
# samples uniformly over [0, cycle_duration].
meshes[-1] = meshes[0]

In [ ]:
disp=[]
print("number: ",number)
# Extra RBF control points, read from dispm/ as mw_<n>.vtp: each carries a set of
# reference positions and a "Displacement" array for that phase. Cells 16 and 19
# concatenate them with the ring-matched points before the interpolation, so they
# add a second, independent source of displacement over whatever region they cover.
#
# Stage 0b (deformation.py) writes these tracked ascending-aorta control points.
# The public phantom deliberately starts from analytic phase surfaces and does not
# attempt a synthetic Stage-0b run, so its dispm/ directory is absent.
#
# The directory is optional. When it is absent the interpolation runs on the
# ring-matched points alone, which is what the synthetic example does — it has no
# upstream tracking stage to supply an equivalent. A study case has the directory
# and is unaffected by this branch.
direct=os.path.join(dir_stl,"dispm")

if not os.path.isdir(direct):
    print(f"no dispm/ at {direct}\n"
          "  -> continuing with the ring-matched points only.")
else:
    infolder_total=os.listdir(direct)


    organized_files={}  
    for file_name in infolder_total:
            if file_name.endswith(".vtp"):
                first_digit = int(file_name.split('_')[1].split('.')[0])  # Extract the first digit
                organized_files.setdefault(first_digit, []).append(file_name)

    load_name_mesh,load_name_mesh_load=[],[] 
        
    for digit in sorted(organized_files.keys(), key=lambda x: f"{x:03}"):
        load_name_mesh.append(organized_files[digit][0])
        load_name_mesh_load.append(os.path.join(direct, organized_files[digit][0]))
    print(sorted(organized_files.keys()))
    for file in load_name_mesh_load:
            mesh_r=pv.read(file)
            disp.append(mesh_r)


def control_points(ring_points, num=None):
    """Ring-matched points for one phase, with the dispm/ points appended if present.

    num=None returns the reference set, in which the dispm/ points are undisplaced;
    otherwise they are displaced by the field stored for phase `num`. With no dispm/
    the ring-matched points are returned unchanged, so the two concatenations that
    cells 16 and 19 perform become no-ops rather than an IndexError.
    """
    if not disp:
        return ring_points
    if num is None:
        return np.concatenate([ring_points, disp[0].points])
    return np.concatenate([ring_points, disp[num].points + disp[num]["Displacement"]])

In [ ]:
orginal_0=meshes[0].copy()

# Trimming planes, from the configuration rather than as literals: they are written
# in the frame of a particular dataset, so nothing outside that frame can use them.
# Their values belong to each input coordinate frame and have no public default.
point_cut  = cfg["trim_split_origin"]
split_norm = cfg["trim_split_normal"]
z_bot      = cfg["trim_floor_origin"]
floor_norm = cfg["trim_floor_normal"]

for num, mesh in enumerate(meshes):
    mesh_D=mesh.clip(origin=point_cut,normal=split_norm,invert=False,inplace=False )

    mesh_D.clip(origin=z_bot,normal=floor_norm,invert=False,inplace=True)


    mesh_E=mesh.clip(origin=point_cut,normal=split_norm)
    mesh_E.clip(origin=plane_info_0[0],normal=plane_info_0[1],inplace=True)
    mesh_origin=pv.merge([mesh_E,mesh_D])
    # fill_holes

    mesh_origin.clean()
    meshes[num]=mesh_origin

if number=="9":
    Cline=centerline.clip(origin=plane_info_arc[0], normal=plane_info_arc[1])
else:
    Cline=centerline
plotter=pv.Plotter(notebook=False)
plotter.add_mesh(mesh_D,color='blue',opacity=0.1)
plotter.add_mesh(mesh_E,color='red',opacity=0.1)
plotter.add_mesh(meshes[0],color='lightgrey',opacity=0.5)
plotter.add_mesh(Cline,color='red',line_width=5)
plotter.show()

In [ ]:
# Cycle closure was applied when the phase surfaces were loaded.


# 3. Starting with the inlet

### 1 calcualte the RBF of the rest


In [ ]:
def cut_ring(mesh, cpoint, normal1):
    slab = mesh.clip(
        normal=normal1,
        origin=cpoint + normal1 * 0.5,
        invert=True
    ).clip(
        normal=normal1,
        origin=cpoint - normal1 * 0.5,
        invert=False
    )

    if slab.n_points == 0:
        return None

    conn = slab.connectivity()
    region_ids = np.unique(conn['RegionId'])

    best_region = None
    best_dist = np.inf

    for rid in region_ids:
        region = conn.threshold(
            [rid, rid],
            scalars='RegionId'
        )
        centroid = region.center
        d = np.linalg.norm(centroid - cpoint)

        if d < best_dist:
            best_dist = d
            best_region = region

    return best_region


In [ ]:
# select points to mesure the rings
distance_min=5
# list_ignore is supplied by the private case configuration. It identifies rings
# that intersect branch ostia and therefore cannot be used as RBF controls.
ring_centers=[]

for num,i in enumerate(Cline.points):
    if ring_centers==[]:
        ring_centers.append(i)
    elif np.min(np.linalg.norm(np.array(ring_centers)-i,axis=1))>distance_min:
            ring_centers.append(i)

for idx in sorted(list_ignore, reverse=True):
    if idx < len(ring_centers):
        ring_centers.pop(idx)
        
ring_normals=[]     
ring_0=[]   

for num,i in enumerate(ring_centers):
    d = np.linalg.norm(Cline.points - i, axis=1)
    d[d <= 0.01] = np.inf  # exclude self
    
    direction_vector = Cline.points[np.argmin(d)] - i
    ring_normals.append(direction_vector / np.linalg.norm(direction_vector))

    ring=cut_ring(meshes[0],i, ring_normals[num])
    if ring is not None: ring_0.append(ring)

plotter = pv.Plotter(notebook=False  )
plotter.add_mesh(meshes[0], color='lightblue', opacity=0.1)
for i,ring in enumerate(ring_0):

    # if i==25:
        plotter.add_mesh(ring_0[i], color='red',opacity=0.5)
        plotter.add_points(ring_centers[i], color='black', point_size=10)
        plotter.add_point_labels(ring_centers[i], [f"{i}"], font_size=36, text_color='black', point_color='yellow', point_size=20, shape_opacity=0.5)
plotter.add_points(Cline.points,color='red',line_width=5)
plotter.show()



In [ ]:
def distance_min(t,mesh, cpoint,vector):
    t_scalar = float(np.asarray(t).reshape(-1)[0])
    moving_point = cpoint + t_scalar * vector
    distances_t=np.linalg.norm(mesh.points - moving_point, axis=1)
    closest_indices = np.argsort(distances_t)[:3]
    closest_points = mesh.points[closest_indices]
    distances=np.linalg.norm(closest_points - moving_point, axis=1)
    return  np.mean(distances)

def find_optimal_point_intrs(mesh, cpoint,vector):
    
    distances=np.linalg.norm(mesh.points - cpoint, axis=1)
    init_t = np.mean(distances)
    
    max_Dist=init_t*1.5

    result = minimize(distance_min, init_t, args=(mesh, cpoint,vector), method='BFGS')
    t_value = float(np.asarray(result.x).reshape(-1)[0])
    dist_final=distance_min(t_value,mesh, cpoint,vector)
    # plot_mesh(mesh1=mesh,c_points1=closest_points,c_points2=np.array( cpoint + result.x * vector))
    if dist_final< max_Dist:
        return np.asarray(cpoint) + t_value * np.asarray(vector), t_value
    return None, None



In [ ]:
# original mesh points section
vectors_rings=[]
point_matching=[]

plotter=pv.Plotter(notebook=False)
for point,i in enumerate(ring_centers):
        plotter.add_mesh(ring_0[point], color='red',opacity=0.5)
        plotter.add_points(ring_centers[point], color='black', point_size=10)
        vectors_center=calculate_vectors(ring_normals[point],first_vector=np.array([0,0,1]))
        vectors_rings.append(vectors_center)
        point_matching_temp=[]
        
        for ray_index, vector in enumerate(vectors_center): 
            points_finded,_=find_optimal_point_intrs(ring_0[point], ring_centers[point],vector)
            if points_finded is None:
                raise RuntimeError(f"ray intersection failed for reference ring {point}, ray {ray_index}")
            point_matching_temp.append(points_finded)
            # plotter.add_mesh(pv.Arrow(start=ring_centers[point],direction=vector,scale=3), color='green')
        point_matching.append(point_matching_temp)
        plotter.add_points(np.array(point_matching[point]), color='green', point_size=10)
        

plotter.show()


In [ ]:
# other mesh selection
number_to_plot=30

point_matching_def=[]
point_matching_def.append(point_matching)

number_to_plot=7
for num, mesh in enumerate(meshes):
    print("number: ",num)
    if num!=0:
        if num==number_to_plot:plotter=pv.Plotter(notebook=False)
        point_matching_def_temp2=[]
        for point,i in enumerate(ring_centers):
            
            if num==number_to_plot:plotter.add_mesh(ring_0[point], color='red',opacity=0.5)
            if num==number_to_plot:plotter.add_points(ring_centers[point], color='black', point_size=10)
            
            
            def_ring=cut_ring(mesh, ring_centers[point], ring_normals[point])
            
            if num==number_to_plot:plotter.add_mesh(def_ring, color='blue',opacity=0.5)
            

            point_matching_def_temp=[]
            for num_vector,vector in enumerate(vectors_rings[point]): 
                
                    points_finded_def,_=find_optimal_point_intrs(def_ring, ring_centers[point],vector)
                    if points_finded_def is None:
                        raise RuntimeError(
                            f"ray intersection failed for phase {num}, ring {point}, ray {num_vector}"
                        )
                    point_matching_def_temp.append(points_finded_def)
                    if num==number_to_plot:plotter.add_mesh(pv.Arrow(start=ring_centers[point],direction=vector,scale=3), color='green')
            point_matching_def_temp2.append(point_matching_def_temp)
            
        point_matching_def.append(point_matching_def_temp2)

        for i in point_matching_def[0]:
            if num==number_to_plot:plotter.add_points(np.array(i), color='green', point_size=5)
        
        for i in point_matching_def[num]:
            if num==number_to_plot:plotter.add_points(np.array(i), color='yellow', point_size=5)
            
        if num==number_to_plot:
            plotter.show()


point_matching_def=np.array(point_matching_def)

In [ ]:
# arr_flat = point_matching_def[0].reshape(-1, 3)

# print(arr_flat.shape)
# print((disp[0].points + disp[0]["Displacement"]).shape)

# newnew = np.concatenate([arr_flat, disp[0].points + disp[0]["Displacement"]])

# 9 save projected points

In [ ]:
mesh_ref_full=pv.read(dir_mesh_full)
deformation_fin=[None]*len(meshes)
deformation_points_temp_zero=control_points(point_matching_def[0].reshape(-1, 3))

for num,mesh in enumerate(meshes):
    arr_flat = point_matching_def[num].reshape(-1, 3)
    
    deformation_points_temp=control_points(arr_flat, num)
    
    
    
    deformation_fin[num]=rbf_calculation_def(mesh_ref_full,deformation_points_temp,deformation_points_temp_zero)

deformation = np.array(deformation_fin)
meshes_new=[None]*len(meshes)

for num,mesh in enumerate(meshes):
    mesh_points=mesh_ref_full.copy()
    deformed_mesh_points = np.stack([mesh_points.points[:,0]+deformation[num][:,0],mesh_points.points[:,1]+deformation[num][:,1],mesh_points.points[:,2]+deformation[num][:,2]], axis=-1)
    mesh_points.points = deformed_mesh_points
    meshes_new[num]=mesh_points

In [ ]:
def save_deformation_vtp(dir,deformation_vectors,mesh_0,new_percentage):

    os.makedirs(dir, exist_ok=True)

    if len(deformation_vectors) == 0:
        return

    first_item = deformation_vectors[0]
    if hasattr(first_item, "save") and hasattr(first_item, "points"):
        for i, mesh in enumerate(deformation_vectors):
            name = "mw_{}.vtp".format(new_percentage[i])
            filename = os.path.join(dir, name)
            mesh.save(filename)
        return
    print("create ply")
    polydata = pv.PolyData(mesh_0.points,mesh_0.faces)

    # Add deformation vectors as point data
    for i in range(len(deformation_vectors)):
        displacement = np.asarray(deformation_vectors[i])
        if displacement.shape[0] != polydata.n_points:
            raise ValueError(
                f"Number of scalars ({displacement.shape[0]}) must match the number of mesh points ({polydata.n_points})."
            )

        polydata["Displacement"] = displacement
        # polydata["Jacobian"] = jacobian_matrices[i]
        name= "mw_{}.vtp".format((new_percentage[i]))
        filename = os.path.join(dir,name)
        for attempt in range(5):
            try:
                polydata.save(filename)
                break
            except OSError as error:
                if attempt == 4 or "Resource temporarily unavailable" not in str(error):
                    raise
                time.sleep(0.2)
save_deformation_vtp((cfg["deformed_mesh_dir"]),deformation,mesh_ref_full,range(0, len(deformation)))

In [ ]:

# def calculate_volume(mesh):
#     original_mesh_cleaned = mesh.clean()
#     original_mesh_cleaned=original_mesh_cleaned.fill_holes(1000).triangulate()
#     return original_mesh_cleaned.volume

# volume=[]
# for mesh in meshes_new:
#     volume.append(calculate_volume(mesh))
# volume=np.array(volume)
# print("volume maximum: ",np.max(volume))
# print("volume minimum: ",np.min(volume))
# print("volume variation: ",np.max(volume)-np.min(volume))

In [ ]:
num_time_steps = len(meshes)
time_steps = [i * time_size / (num_time_steps - 1) for i in range(num_time_steps)]

output_file =(cfg["wall_motion_out_dir"])
    
if not os.path.exists(output_file):
    os.makedirs(output_file)
    

for surf_name in out_name:
    # No leading "\\": joined with os.path.join below, so this resolves off
    # Windows too. A leading separator would make the name absolute and the
    # output land at the file-system root.
    file_name_save=f"wall_motion_{surf_name}_{number}.txt"
    print(f"Processing surface: {file_name_save}")
    mes = main_dir + surf_name
    mesh_ref=pv.read(mes)
    deformation_fin=[None]*len(meshes)

    deformation_points_temp_zero=control_points(point_matching_def[0].reshape(-1, 3))


    for num,mesh in enumerate(meshes):
        
        arr_flat = point_matching_def[num].reshape(-1, 3)
        deformation_points_temp=control_points(arr_flat, num)

        if num==len(meshes)-1:
            deformation_fin[num]=deformation_fin[0]
        else:
                deformation_fin[num]=rbf_calculation_def(mesh_ref,deformation_points_temp,deformation_points_temp_zero)

    deformation = np.array(deformation_fin)
    

    num_points = deformation.shape[1]

    # Interpolation setup
    num_interpolated_steps = 100
    original_time = np.array(time_steps)
    interpolated_time = np.linspace(original_time[0], original_time[-1], num_interpolated_steps)

    
    
    with open(os.path.join(output_file, file_name_save), 'w') as f:
        # Write header: time_steps, num_points, total_values
        f.write(f"3 {num_interpolated_steps} {num_points} \n\n")
        
        # Write time steps
        for t in interpolated_time:
            f.write(f"{t:.4f}\n")
        f.write("\n")
        
        # Write deformation data for each point
        for point_idx in range(num_points):
            f.write(f"\n{mesh_ref.point_data['GlobalNodeID'][point_idx]}\n")

            point_deformation = deformation[:, point_idx, :] * 0.1  # to cm

            # Create interpolation functions for dx, dy, dz
            interp_dx = interp1d(original_time, point_deformation[:, 0], kind='cubic')
            interp_dy = interp1d(original_time, point_deformation[:, 1], kind='cubic')
            interp_dz = interp1d(original_time, point_deformation[:, 2], kind='cubic')

            # Get interpolated values
            interpolated_dx = interp_dx(interpolated_time)
            interpolated_dy = interp_dy(interpolated_time)
            interpolated_dz = interp_dz(interpolated_time)
            
            for time_idx in range(num_interpolated_steps):
                dx, dy, dz = interpolated_dx[time_idx], interpolated_dy[time_idx], interpolated_dz[time_idx]
                f.write(f"{dx:.5f} {dy:.5f} {dz:.5f}\n")

In [ ]:
# #interface

# deformation_fin=[None]*len(meshes)

# deformation_points_temp_zero=np.concatenate([point_matching_def[0].reshape(-1, 3), (disp[0].points)])


# for num,mesh in enumerate(meshes):
    
#     arr_flat = point_matching_def[num].reshape(-1, 3)
#     deformation_points_temp=np.concatenate([arr_flat, (disp[num].points + disp[num]["Displacement"])])
#     deformation_fin[num]=rbf_calculation_def(mesh_ref,deformation_points_temp,deformation_points_temp_zero)
    

# deformation = np.array(deformation_fin)

# deformation[len(deformation)-1]=deformation[0]
# meshes_new=[None]*len(meshes)

# for num,mesh in enumerate(meshes):

    
#     mesh_points=mesh_ref.copy()
#     deformed_mesh_points = np.stack([mesh_points.points[:,0]+deformation[num][:,0],mesh_points.points[:,1]+deformation[num][:,1],mesh_points.points[:,2]+deformation[num][:,2]], axis=-1)
#     mesh_points.points = deformed_mesh_points
#     meshes_new[num]=mesh_points


